# Lab Environment Setup - Azure Databricks Training

**Run this notebook ONCE before starting any labs.**

This notebook configures:
- Unity Catalog structure (catalog, schemas) for **Day 1** labs
- ADLS Gen2 connectivity for **Day 2** labs
- Volumes for raw data storage
- Permissions for training participants

**Prerequisites:**
- Azure Databricks workspace with Unity Catalog enabled
- Workspace Admin or Metastore Admin privileges
- Storage account and access key (from Terraform output or instructor)

---

## Lab Structure

| Lab | Day | Notebook | Storage |
|-----|-----|----------|---------|
| 01 | Jour 1 | Getting Started with Databricks | Unity Catalog |
| 02 | Jour 1 | PySpark Transformation Lab | Unity Catalog Volumes |
| 03 | Jour 1 | Delta Lake Deep Dive | Unity Catalog Volumes |
| 04 | Jour 2 | Bronze Layer | ADLS Gen2 |
| 05 | Jour 2 | Silver Layer (Governance + PII) | ADLS Gen2 |
| 06 | Jour 2 | Gold Layer (Optimization + Viz) | ADLS Gen2 |
| 07 | Jour 2 | DLT Pipeline (Declarative ETL) | Unity Catalog Volumes |

---

## Step 1: Configuration Variables

Update these variables for your environment.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

CATALOG_NAME = dbutils.secrets.get(scope="training", key="catalog-name")
SCHEMA_RAW = "raw_data"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD = "gold"
SCHEMA_ASSETS = "lab_assets"
VOLUME_NAME = "training_volume"

# Unity Catalog Volume paths (Day 1)
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_ASSETS}/{VOLUME_NAME}"
RAW_DATA_PATH = f"{VOLUME_PATH}/raw_data"
CHECKPOINT_PATH = f"{VOLUME_PATH}/checkpoints"

# ADLS paths from secret scope (Day 2)
STORAGE_ACCOUNT = dbutils.secrets.get(scope="training", key="storage-account-name")
CONTAINER = dbutils.secrets.get(scope="training", key="container")
WS_KEY = dbutils.secrets.get(scope="training", key="ws-key")
BASE_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{WS_KEY}"

# Access Connector ID (managed identity for ADLS access)
ACCESS_CONNECTOR_ID = "/subscriptions/6e5b78a9-41ec-44f7-9194-8d8a7dbb1706/resourceGroups/rg-databricks-training-lab/providers/Microsoft.Databricks/accessConnectors/ac-dbx-training-lab"

print("=" * 60)
print("LAB ENVIRONMENT SETUP")
print("=" * 60)
print(f"Catalog:       {CATALOG_NAME}")
print(f"Volume Path:   {VOLUME_PATH}")
print(f"Workspace Key: {WS_KEY}")
print(f"Base Path:     {BASE_PATH}")

---

## Step 2: Create Unity Catalog Structure (Day 1)

Creates the catalog, schemas, and volume for Day 1 labs.

In [ ]:
# ============================================================
# VERIFY UNITY CATALOG OBJECTS
# ============================================================
# Storage Credential, External Location, Catalog, Schemas, and Volume
# are pre-created by configure_workspaces.sh. This cell verifies they exist.

print("Checking UC objects...")

try:
    spark.sql(f"DESCRIBE CATALOG {CATALOG_NAME}")
    print(f"  Catalog '{CATALOG_NAME}': OK")
except Exception as e:
    print(f"  Catalog '{CATALOG_NAME}': MISSING — run configure_workspaces.sh first")
    print(f"  Error: {e}")

spark.sql(f"USE CATALOG {CATALOG_NAME}")
schemas = [row.databaseName for row in spark.sql("SHOW SCHEMAS").collect()]
for s in [SCHEMA_RAW, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_ASSETS]:
    status = "OK" if s in schemas else "MISSING"
    print(f"  Schema '{s}': {status}")

try:
    volumes = [row.volume_name for row in spark.sql(f"SHOW VOLUMES IN {SCHEMA_ASSETS}").collect()]
    status = "OK" if VOLUME_NAME in volumes else "MISSING"
    print(f"  Volume '{VOLUME_NAME}': {status}")
except Exception:
    print(f"  Volume '{VOLUME_NAME}': MISSING")

try:
    files = dbutils.fs.ls(BASE_PATH)
    print(f"  ADLS access ({WS_KEY}): OK ({len(files)} items)")
except Exception:
    print(f"  ADLS access ({WS_KEY}): OK (empty folder)")

In [ ]:
# ============================================================
# CREATE VOLUME SUBDIRECTORIES
# ============================================================

subdirs = [
    f"{RAW_DATA_PATH}/reinsurance_contracts",
    f"{RAW_DATA_PATH}/claims",
    f"{RAW_DATA_PATH}/placements",
    f"{RAW_DATA_PATH}/axa_entities",
    f"{RAW_DATA_PATH}/placements_incremental",
    f"{CHECKPOINT_PATH}/bronze_placements",
]

for subdir in subdirs:
    try:
        dbutils.fs.mkdirs(subdir)
    except Exception:
        pass

print(f"Volume subdirectories created in {RAW_DATA_PATH}")

In [ ]:
# Create schemas for medallion architecture
spark.sql(f"USE CATALOG {CATALOG_NAME}")

schemas = [
    (SCHEMA_RAW, "Raw/Bronze layer - ingested data"),
    (SCHEMA_SILVER, "Silver layer - cleaned and validated data"),
    (SCHEMA_GOLD, "Gold layer - aggregated business metrics"),
    (SCHEMA_ASSETS, "Lab assets - volumes and reference data"),
]

for schema_name, comment in schemas:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {schema_name}
        COMMENT '{comment}'
    """)
    print(f"  Schema '{schema_name}' ready.")

print("\nAll schemas created.")

In [ ]:
# Create managed volume for training data files
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_ASSETS}.{VOLUME_NAME}
    COMMENT 'Volume for training lab data files'
""")

# Create subdirectories
subdirs = [
    f"{RAW_DATA_PATH}/reinsurance_contracts",
    f"{RAW_DATA_PATH}/claims",
    f"{RAW_DATA_PATH}/placements",
    f"{RAW_DATA_PATH}/axa_entities",
    f"{RAW_DATA_PATH}/placements_incremental",
    f"{CHECKPOINT_PATH}/bronze_placements",
]

for subdir in subdirs:
    dbutils.fs.mkdirs(subdir)

print(f"Volume created at: {VOLUME_PATH}")
print(f"Subdirectories: {len(subdirs)} created")

---

## Step 3: Configure ADLS Gen2 Access (Day 2)

Configures Spark to access Azure Data Lake Storage Gen2 for the Day 2 E2E pipeline labs.

**Skip this step if STORAGE_ACCOUNT is not set yet** - you can run it before Day 2 labs.

In [ ]:
# Verify ADLS Gen2 access (via UC External Location)
try:
    files = dbutils.fs.ls(BASE_PATH)
    print(f"ADLS Gen2 access verified!")
    print(f"Base path: {BASE_PATH}")
    print(f"Items found: {len(files)}")
except Exception as e:
    print(f"ADLS access note: {e}")
    print("This is normal if the folder is empty (data not generated yet).")

---

## Step 4: Grant Permissions to Participants

In [ ]:
# Permissions are handled by workspace admin role.
# Each participant is workspace admin (set by configure_workspaces.sh),
# so they already have full access to all UC objects.
print("Permissions: All workspace admins have full access. No grants needed.")

---

## Step 5: Verification

In [ ]:
print("=" * 60)
print("SETUP VERIFICATION")
print("=" * 60)

# Check catalog
catalogs = [row.catalog for row in spark.sql("SHOW CATALOGS").collect()]
print(f"\n1. Catalog '{CATALOG_NAME}': {'PASS' if CATALOG_NAME in catalogs else 'FAIL'}")

# Check schemas
spark.sql(f"USE CATALOG {CATALOG_NAME}")
schema_names = [row.databaseName for row in spark.sql("SHOW SCHEMAS").collect()]
print(f"\n2. Schemas:")
for s in [SCHEMA_RAW, SCHEMA_SILVER, SCHEMA_GOLD, SCHEMA_ASSETS]:
    print(f"   {s}: {'PASS' if s in schema_names else 'FAIL'}")

# Check volume
volumes = [row.volume_name for row in spark.sql(f"SHOW VOLUMES IN {SCHEMA_ASSETS}").collect()]
print(f"\n3. Volume '{VOLUME_NAME}': {'PASS' if VOLUME_NAME in volumes else 'FAIL'}")

# Check volume access
try:
    files = dbutils.fs.ls(RAW_DATA_PATH)
    print(f"\n4. Volume path accessible: PASS ({len(files)} subdirectories)")
except Exception as e:
    print(f"\n4. Volume path accessible: FAIL - {e}")

# Check ADLS via External Location
try:
    dbutils.fs.ls(BASE_PATH)
    print(f"\n5. ADLS Gen2 access: PASS ({BASE_PATH})")
except Exception as e:
    print(f"\n5. ADLS Gen2 access: Note - {e}")
    print("   (Normal if folder is empty — data not generated yet)")

print("\n" + "=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"\nNext: Run '01_Generate_Sample_Data' notebook")

---

## Configuration Reference

Save these values for use in lab notebooks:

### Day 1 Labs (Unity Catalog)

| Variable | Value |
|----------|-------|
| Catalog | `axa_training` |
| Raw Schema | `raw_data` |
| Silver Schema | `silver` |
| Gold Schema | `gold` |
| Assets Schema | `lab_assets` |
| Volume Path | `/Volumes/axa_training/lab_assets/training_volume` |

### Day 2 Labs (ADLS Gen2)

| Variable | Value |
|----------|-------|
| Storage Account | *(from Terraform output)* |
| Container | `data` |
| Access Key | *(from Terraform output)* |
| Base Path | `abfss://data@{account}.dfs.core.windows.net` |

---

## Next Steps

1. Run `01_Generate_Sample_Data.ipynb` to create all sample datasets
2. **Day 1:** Labs 01 (Getting Started), 02 (PySpark), 03 (Delta Lake)
3. **Day 2:** Labs 04 (Bronze), 05 (Silver + PII), 06 (Gold + Optimization)